<a href="https://colab.research.google.com/github/Jyotsna135-bit/GenerativeAI/blob/main/GenerativeAI/Notebooks/Deployment/Ollama/elora_chatbot_documented.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# eLORA FAQ Chatbot using RAG (Ollama)

This notebook builds a chatbot for AERB's eLORA system using the 202 Q&A pairs provided in `eLORA-Jyotsna.xlsx`.

**Approach:** Retrieval-Augmented Generation (RAG) with Ollama, using `nomic-embed-text` for embeddings and `qwen3:0.6b` as the LLM — same models from the embedding notebook, so this stays consistent with the rest of the repo.

**A few design choices come directly from this article on common RAG mistakes:** https://towardsdatascience.com/10-common-rag-mistakes-we-keep-seeing-in-production/

- *Don't rely on cosine similarity alone* — the bot runs a keyword match alongside the embedding match and combines both scores. A pure-embedding search misses exact terms like "Licencee login" or "Forgot Password" that a user might type almost verbatim.
- *Don't dump everything into the prompt* — only the single best-matching Q&A pair is passed to the LLM, not all 202 rows. Keeps the context small and the answer grounded.
- *Don't trust a confident-sounding answer blindly* — every response carries a similarity score. If the score is below a threshold, the bot says it isn't confident instead of guessing.

> Set runtime to **T4 GPU** before running. Upload `eLORA-Jyotsna.xlsx` to the Colab session (left sidebar → Files → upload) before running Step 4.

## Installing Ollama

`zstd` is needed by the installer to unpack itself — install it first or the setup fails partway through.

In [1]:
!apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## Starting the Server

In [2]:
import subprocess, time, requests

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(15):
    try:
        if requests.get("http://localhost:11434").status_code == 200:
            print("ollama is running")
            break
    except:
        time.sleep(1)

ollama is running


## Pulling the Models

In [3]:
!ollama pull nomic-embed-text
!ollama pull qwen3:0.6b

## Step 1 — Load the FAQ Data

Reading the 202 question/answer pairs from the Excel file. Each row becomes one entry in our knowledge base — the `faq_question` is what gets matched against, the `faq_answer` is what gets returned.

In [4]:
!pip install openpyxl pandas -q

In [5]:
import pandas as pd
import re

df = pd.read_excel("eLORA-Jyotsna.xlsx")

# clean up HTML-encoded characters and stray entities that show up in the raw text
def clean_text(text):
    text = str(text)
    text = re.sub(r"&#\d+;", " ", text)   # numeric HTML entities like &#61664;
    text = text.replace("?", "'")          # the source file uses ? in place of curly quotes/arrows
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["faq_question"] = df["faq_question"].apply(clean_text)
df["faq_answer"] = df["faq_answer"].apply(clean_text)

print(f"Loaded {len(df)} Q&A pairs")
df.head(3)

Loaded 202 Q&A pairs


,faq_question,faq_answer
0,"While applying for Licence, installations are ...",Please verify the login details. Licence appli...
1,I'm not able to see Telegamma Therapy equipmen...,"Ans. At present, Telegamma Therapy equipment o..."
2,"I have forgotten my password, how to obtain ne...",Visit eLORA home page and click on 'Forgot Pas...


## Step 2 — Embed Every Question

We embed `faq_question` for all 202 rows once, upfront. This is the knowledge base the chatbot searches against. Doing this once and caching it means every user query only needs one new embedding call, not 202.

In [6]:
import numpy as np
from tqdm import tqdm

def get_embedding(text):
    response = requests.post(
        "http://localhost:11434/api/embeddings",
        json={"model": "nomic-embed-text", "prompt": text}
    )
    return response.json()["embedding"]

print("embedding all FAQ questions, this takes a minute or two...")
question_embeddings = []
for q in tqdm(df["faq_question"].tolist()):
    question_embeddings.append(get_embedding(q))

question_embeddings = np.array(question_embeddings)
print(f"Embedded {len(question_embeddings)} questions, vector dim = {question_embeddings.shape[1]}")

embedding all FAQ questions, this takes a minute or two...


100%|██████████| 202/202 [01:36<00:00,  2.10it/s]

Embedded 202 questions, vector dim = 768


## Step 3 — Hybrid Retrieval (Embedding + Keyword)

Pure embedding search misses exact terms — a user typing "Forgot Password" should match the FAQ about forgotten passwords directly, even if the embedding similarity isn't the absolute highest. So we combine two signals:

1. **Cosine similarity** between the user's question and every FAQ question (captures meaning/paraphrasing)
2. **Keyword overlap** — how many words the user's question shares with each FAQ question (captures exact terms, codes, names)

The final score is a weighted combination. This is the hybrid retrieval approach the reference article recommends instead of relying on cosine similarity alone.

In [7]:
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def keyword_overlap_score(query, candidate):
    # simple word-overlap ratio, ignoring very short/common words
    stopwords = {"the", "is", "a", "an", "to", "of", "in", "for", "on", "and", "i", "my", "how", "do", "does", "will", "can"}
    q_words = set(w.lower() for w in re.findall(r"\w+", query) if w.lower() not in stopwords and len(w) > 2)
    c_words = set(w.lower() for w in re.findall(r"\w+", candidate) if w.lower() not in stopwords and len(w) > 2)
    if not q_words:
        return 0.0
    overlap = q_words.intersection(c_words)
    return len(overlap) / len(q_words)

def retrieve_best_match(user_query, top_k=3, embedding_weight=0.7, keyword_weight=0.3):
    query_embedding = get_embedding(user_query)

    scores = []
    for idx, row in df.iterrows():
        emb_score = cosine_similarity(query_embedding, question_embeddings[idx])
        kw_score = keyword_overlap_score(user_query, row["faq_question"])
        combined = (embedding_weight * emb_score) + (keyword_weight * kw_score)
        scores.append((idx, combined, emb_score, kw_score))

    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

## Step 4 — The Chatbot Function

This ties retrieval and generation together, with a confidence check in between:

1. Retrieve the best-matching FAQ entry using hybrid search
2. If the combined score is below `CONFIDENCE_THRESHOLD`, tell the user directly instead of guessing — this is the "don't trust a confident-sounding answer" fix from the article
3. Otherwise, pass only that one matched Q&A pair to the LLM and ask it to phrase a natural answer from it — not the whole spreadsheet, just the one relevant entry

In [8]:
from openai import OpenAI

!pip install openai -q

llm_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

CONFIDENCE_THRESHOLD = 0.55  # tune this based on testing — lower = more lenient matching

def ask_elora_bot(user_query, verbose=True):
    matches = retrieve_best_match(user_query, top_k=3)
    best_idx, best_score, emb_score, kw_score = matches[0]

    if verbose:
        print(f"Top match (score={best_score:.3f}, embedding={emb_score:.3f}, keyword={kw_score:.3f}):")
        print(f"  FAQ: {df.iloc[best_idx]['faq_question']}\n")

    # confidence check — don't answer if nothing matched well enough
    if best_score < CONFIDENCE_THRESHOLD:
        return {
            "answer": "I'm not confident this question is covered in the eLORA FAQ. Please check the eLORA help section or contact AERB support directly.",
            "confidence": best_score,
            "matched_question": None,
            "found": False
        }

    matched_question = df.iloc[best_idx]["faq_question"]
    matched_answer = df.iloc[best_idx]["faq_answer"]

    # only the single matched Q&A pair goes into the prompt — not the whole sheet
    prompt = f"""You are a helpful assistant for the eLORA system (AERB's radiation licensing portal).
A user asked a question. Use ONLY the FAQ entry below to answer — do not add information that isn't in it.

FAQ Question: {matched_question}
FAQ Answer: {matched_answer}

User's question: {user_query}

Give a clear, direct answer based only on the FAQ entry above."""

    response = llm_client.chat.completions.create(
        model="qwen3:0.6b",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300
    )

    return {
        "answer": response.choices[0].message.content,
        "confidence": best_score,
        "matched_question": matched_question,
        "found": True
    }

## Step 5 — Try It Out

A few test queries — paraphrased versions of real FAQ questions, to check the hybrid retrieval is working and not just matching on exact wording.

In [9]:
test_queries = [
    "I forgot my password, what do I do?",
    "How can I check the status of my application?",
    "How do I update my employer's email address?",
]

for query in test_queries:
    print("=" * 70)
    print(f"USER: {query}\n")
    result = ask_elora_bot(query)
    print(f"BOT: {result['answer']}\n")

USER: I forgot my password, what do I do?

Top match (score=0.705, embedding=0.864, keyword=0.333):
  FAQ: I have forgotten my password, how to obtain new password

BOT: Visit e-LORA Home Page and click on 'Forgot Password'. Provide your 'Username' and 'Registered Email Id' and submit with the Captcha. You will receive new password on your registered email address as well as on registered mobile no. via SMS (applicable for Institute as well as Radiation Professional accounts).

USER: How can I check the status of my application?

Top match (score=0.871, embedding=0.815, keyword=1.000):
  FAQ: I have submitted application form for Institute registration, how to check its status'

BOT: To check the status of your application, please use the "Application Status" option on the eLORA home page. This will provide you with the status based on the submitted form. After review, approval or rejection emails will be sent to the Employer mentioned in the application form.

USER: How do I update my

## Step 6 — Testing the Confidence Threshold

This is the "don't trust the LLM blindly" check — asking something clearly outside the FAQ's scope should trigger the low-confidence fallback, not a hallucinated answer.

In [10]:
off_topic_query = "What is the recipe for chocolate cake?"

result = ask_elora_bot(off_topic_query)
print(f"USER: {off_topic_query}\n")
print(f"BOT: {result['answer']}")
print(f"Found in FAQ: {result['found']}")

Top match (score=0.359, embedding=0.406, keyword=0.250):
  FAQ: I want to submit site and layout application. What are the the guidelines for preparation of radiotherapy site and layout drawings.

USER: What is the recipe for chocolate cake?

BOT: I'm not confident this question is covered in the eLORA FAQ. Please check the eLORA help section or contact AERB support directly.
Found in FAQ: False


## Step 7 — Interactive Chat Loop

Run this cell to chat with the bot interactively in Colab. Type `exit` to stop.

In [11]:
print("eLORA FAQ Chatbot — type 'exit' to quit\n")

while True:
    user_input = input("You: ")
    if user_input.strip().lower() == "exit":
        print("Bot: Goodbye!")
        break
    result = ask_elora_bot(user_input, verbose=False)
    print(f"Bot: {result['answer']}")
    print(f"   (confidence: {result['confidence']:.2f})\n")

eLORA FAQ Chatbot — type 'exit' to quit

You: i frogot my password , what do i do 
Bot: Visit eLORA's homepage and click on "Forgot Password". Provide your "Username" and "Registered Email Id" and submit with a Captcha. You will receive a new password on your registered email address as well as on your registered mobile number via SMS.
   (confidence: 0.60)

You: eexit 
Bot: I'm not confident this question is covered in the eLORA FAQ. Please check the eLORA help section or contact AERB support directly.
   (confidence: 0.33)

You: exit 
Bot: Goodbye!


## Notes on Design Choices

A few things worth calling out, tying back to the reference article on RAG mistakes:

- **Hybrid retrieval, not pure cosine similarity.** The eLORA FAQ has specific terms — "Licencee login", "Forgot Password", form names — that a user is likely to type close to verbatim. Pure embedding similarity can rank a paraphrased-but-unrelated question above the exact match. Combining keyword overlap with embedding similarity fixes this.

- **One matched Q&A pair in the prompt, not all 202.** Feeding the entire spreadsheet to the LLM on every query would be slow, expensive, and more likely to cause the model to mix up answers across rows. Retrieval narrows it down first; generation only sees the one relevant entry.

- **Confidence threshold instead of trusting the LLM's judgment.** If retrieval returns nothing with a good match score, the bot says so plainly instead of asking the LLM to generate something from a weak match. This avoids the "not in the chunks" vs "not in the corpus" gap the article describes — here it's simpler since there's no corpus beyond the FAQ, but the same principle holds: don't let the LLM paper over a retrieval miss.

- **`CONFIDENCE_THRESHOLD = 0.55` is a starting point**, not a fixed answer. Worth testing with more real user queries and adjusting up or down based on false positives/negatives.

**Possible improvements for later:** logging every query + matched FAQ + confidence score for review, a feedback button (was this answer helpful?), and expanding the keyword matcher to handle eLORA-specific abbreviations.